# Label representation versus predictive performance

For one classifier artifact, test whether labels with scarce or weakly represented positive training samples have poorer held-out performance.

In [ ]:
from copy import deepcopy
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from sklearn.metrics import (
    average_precision_score,
    precision_recall_fscore_support,
    roc_auc_score,
)
from torch.utils.data import DataLoader

In [ ]:
ROOT = Path.cwd().resolve()
if not (ROOT / "artifacts").exists():
    ROOT = ROOT.parent
if not (ROOT / "artifacts").exists():
    raise FileNotFoundError("Run this notebook from the repository or notebooks directory")
sys.path.insert(0, str(ROOT))

from src.data import fit_standardizer, make_dataset, split_indices
from src.hashing import calculate_run_id
from src.model import build_model

## Select an artifact

In [ ]:
PROJECT_BASENAME = "bmra_student_project"
RUN_ID = "ddf044ae55d4"

artifact_dir = ROOT / "artifacts" / PROJECT_BASENAME / RUN_ID
framework_path = artifact_dir / "framework.yaml"
project_path = artifact_dir / f"{PROJECT_BASENAME}.yaml"
model_path = artifact_dir / "model.pt"
metrics_path = artifact_dir / "metrics.json"
for path in (framework_path, project_path, model_path, metrics_path):
    if not path.exists():
        raise FileNotFoundError(path)
artifact_dir

In [ ]:
framework = yaml.safe_load(framework_path.read_bytes())
project = yaml.safe_load(project_path.read_bytes())
metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
cv_metrics = metrics.get("cross_validation", {})
selected_framework = deepcopy(framework)
selected_candidate = cv_metrics.get("selected_candidate", {})
for section in ("model", "training"):
    selected_framework[section].update(selected_candidate.get(section, {}))
resolved_config = {**framework, "project": {**framework["project"], **project}}
calculated_run_id = calculate_run_id(resolved_config)
if calculated_run_id != RUN_ID:
    raise ValueError(f"Artifact/config hash mismatch: {RUN_ID} != {calculated_run_id}")
print(f"Assessing {PROJECT_BASENAME}/{RUN_ID}")

## Reconstruct test predictions

In [ ]:
frame = pd.read_parquet(ROOT / framework["data"]["path"])
feature_columns = project["features"]
label_columns = project["labels"]
missing_columns = set(feature_columns + label_columns) - set(frame.columns)
if missing_columns:
    raise ValueError(f"Columns missing from evaluation data: {sorted(missing_columns)}")

split = framework["split"]
train_idx, val_idx, test_idx = split_indices(
    len(frame),
    split["train"],
    split["val"],
    split["test"],
    split["seed"],
    frame=frame,
    label_columns=label_columns,
    strategy=split.get("strategy", "random"),
    group_column=split.get("group_column"),
)
development_idx = np.sort(np.concatenate([train_idx, val_idx]))
fit_idx = development_idx if cv_metrics.get("enabled") else train_idx
means, scales = fit_standardizer(frame.iloc[fit_idx], feature_columns)
test_data = make_dataset(frame.iloc[test_idx], feature_columns, label_columns, means, scales)
test_loader = DataLoader(test_data, batch_size=framework["training"]["batch_size"])
print(
    f"Full dataset: {len(frame):,} rows; test split: {len(test_data):,} rows; "
    f"strategy: {split.get('strategy', 'legacy random')}"
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_model(selected_framework, project).to(device)
model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
model.eval()

all_probabilities, all_targets, all_masks = [], [], []
with torch.no_grad():
    for features, targets, feature_mask, label_mask in test_loader:
        logits = model(features.to(device), feature_mask.to(device))
        all_probabilities.append(torch.sigmoid(logits).cpu())
        all_targets.append(targets)
        all_masks.append(label_mask.bool())

probabilities = torch.cat(all_probabilities).numpy()
targets = torch.cat(all_targets).numpy().astype(int)
availability = torch.cat(all_masks).numpy()
threshold_by_label = cv_metrics.get("label_thresholds", {})
decision_thresholds = np.array(
    [threshold_by_label.get(label, 0.5) for label in label_columns]
)
predictions = probabilities >= decision_thresholds
probabilities.shape

## Per-label representation and performance

Representation is measured only on the training split. Accuracy, F1, average precision, and ROC-AUC use observed test labels.

In [ ]:
train_features = frame.iloc[fit_idx][feature_columns].to_numpy(dtype=float)
train_labels = frame.iloc[fit_idx][label_columns].to_numpy(dtype=float)
num_features = len(feature_columns)
observed_feature_values = train_features[~np.isnan(train_features)]
if not np.isin(observed_feature_values, [0, 1]).all():
    raise ValueError("Feature-space coverage metrics require binary features")

def feature_rates(values):
    observed = ~np.isnan(values)
    return np.divide(
        np.nansum(values, axis=0),
        observed.sum(axis=0),
        out=np.full(values.shape[1], np.nan),
        where=observed.sum(axis=0) > 0,
    )

rows = []
for index, label in enumerate(label_columns):
    observed = availability[:, index]
    y_true = targets[observed, index]
    y_score = probabilities[observed, index]
    y_pred = predictions[observed, index]
    accuracy = (
        (y_pred == y_true).mean()
        if observed.any()
        else np.nan
    )
    if observed.any():
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average="binary", zero_division=0
        )
    else:
        precision = recall = f1 = np.nan
    has_both_test_classes = len(np.unique(y_true)) == 2

    train_observed = ~np.isnan(train_labels[:, index])
    train_positive = train_observed & (train_labels[:, index] == 1)
    train_negative = train_observed & (train_labels[:, index] == 0)
    positive_features = train_features[train_positive]
    negative_features = train_features[train_negative]
    observed_positive_features = ~np.isnan(positive_features)
    positive_active = (positive_features == 1) & observed_positive_features
    positive_count = int(train_positive.sum())

    if positive_count:
        observed_feature_count = observed_positive_features.sum()
        positive_density = (
            positive_active.sum() / observed_feature_count * 100
            if observed_feature_count
            else np.nan
        )
        positive_coverage = positive_active.any(axis=0).mean() * 100
        profiles = np.nan_to_num(positive_features, nan=-1.0)
        unique_profile_count = np.unique(profiles, axis=0).shape[0]
        profile_diversity = unique_profile_count / positive_count * 100
    else:
        unique_profile_count = 0
        positive_density = positive_coverage = profile_diversity = np.nan

    positive_rates = feature_rates(positive_features)
    negative_rates = feature_rates(negative_features)
    prevalence_gap = np.abs(positive_rates - negative_rates)
    finite_gaps = prevalence_gap[np.isfinite(prevalence_gap)]
    rows.append(
        {
            "label": label,
            "missing_pct": frame[label].isna().mean() * 100,
            "test_missing_pct": (1 - observed.mean()) * 100,
            "test_observed": int(observed.sum()),
            "test_positive_count": int(y_true.sum()),
            "test_negative_count": int(len(y_true) - y_true.sum()),
            "test_accuracy": accuracy,
            "test_precision": precision,
            "test_recall": recall,
            "test_f1": f1,
            "test_roc_auc": (
                roc_auc_score(y_true, y_score) if has_both_test_classes else np.nan
            ),
            "test_average_precision": (
                average_precision_score(y_true, y_score)
                if has_both_test_classes
                else np.nan
            ),
            "train_positive_count": positive_count,
            "train_positive_prevalence_pct": (
                positive_count / train_observed.sum() * 100 if train_observed.any() else np.nan
            ),
            "positive_feature_density_pct": positive_density,
            "positive_feature_coverage_pct": positive_coverage,
            "unique_positive_profiles": unique_profile_count,
            "positive_profile_diversity_pct": profile_diversity,
            "mean_feature_prevalence_gap": (
                finite_gaps.mean() if len(finite_gaps) else np.nan
            ),
            "signal_features_10pct": int((finite_gaps >= 0.10).sum()),
        }
    )
label_assessment = pd.DataFrame(rows).set_index("label")
label_assessment.sort_values(["missing_pct", "test_accuracy"], ascending=[False, True]).head(20)

In [ ]:
valid = label_assessment.dropna(subset=["missing_pct", "test_accuracy"])
if valid["missing_pct"].nunique() > 1:
    pearson = valid["missing_pct"].corr(valid["test_accuracy"], method="pearson")
    spearman = valid["missing_pct"].rank().corr(valid["test_accuracy"].rank())
else:
    pearson = np.nan
    spearman = np.nan
    print("Correlation is undefined: every configured label has the same missing percentage.")
pd.Series(
    {
        "labels": len(valid),
        "minimum_missing_pct": valid["missing_pct"].min(),
        "maximum_missing_pct": valid["missing_pct"].max(),
        "pearson_correlation": pearson,
        "spearman_correlation": spearman,
    },
    name="value",
)

## Missingness–accuracy scatter plot

In [ ]:
fig, axis = plt.subplots(figsize=(9, 6))
points = axis.scatter(
    valid["missing_pct"],
    valid["test_accuracy"],
    c=valid["test_positive_count"],
    cmap="viridis",
    alpha=0.75,
    edgecolor="black",
    linewidth=0.3,
)
if valid["missing_pct"].nunique() > 1:
    slope, intercept = np.polyfit(valid["missing_pct"], valid["test_accuracy"], 1)
    x_line = np.linspace(valid["missing_pct"].min(), valid["missing_pct"].max(), 100)
    axis.plot(x_line, slope * x_line + intercept, color="crimson", linestyle="--")
    correlation_text = f"Pearson r = {pearson:.3f}; Spearman ρ = {spearman:.3f}"
else:
    correlation_text = "Correlation undefined: missingness has no variation"

axis.text(0.02, 0.03, correlation_text, transform=axis.transAxes)
axis.set(
    xlabel="Missing label values in full dataset (%)",
    ylabel="Test accuracy on observed values",
    title=f"Label missingness versus accuracy: {PROJECT_BASENAME}/{RUN_ID}",
)
axis.grid(alpha=0.2)
fig.colorbar(points, ax=axis, label="Positive test examples")
plt.tight_layout()
plt.show()

## Does positive-sample representation explain ROC-AUC?

The following diagnostics are descriptive, not causal. Positive count and prevalence measure sample scarcity; feature density and coverage measure how much of the binary feature space is active among positive training cases; profile diversity measures distinct positive feature patterns; and prevalence gap measures separation from negative training cases. ROC-AUC is omitted for labels without both classes in the test split.

In [ ]:
representation_metrics = {
    "Positive training samples": "train_positive_count",
    "Positive training prevalence (%)": "train_positive_prevalence_pct",
    "Active-feature density in positives (%)": "positive_feature_density_pct",
    "Feature coverage by positives (%)": "positive_feature_coverage_pct",
    "Unique positive feature profiles": "unique_positive_profiles",
    "Unique positive profiles (%)": "positive_profile_diversity_pct",
    "Mean positive-negative prevalence gap": "mean_feature_prevalence_gap",
    "Features with at least 10% prevalence gap": "signal_features_10pct",
}
MIN_TEST_CLASS_COUNT = 5
auc_defined = label_assessment.dropna(subset=["test_roc_auc"]).copy()
auc_labels = auc_defined.loc[
    (auc_defined["test_positive_count"] >= MIN_TEST_CLASS_COUNT)
    & (auc_defined["test_negative_count"] >= MIN_TEST_CLASS_COUNT)
].copy()
correlation_rows = []
for description, column in representation_metrics.items():
    values = auc_labels[[column, "test_roc_auc"]].dropna()
    if values[column].nunique() > 1:
        pearson_auc = values[column].corr(values["test_roc_auc"])
        spearman_auc = values[column].rank().corr(values["test_roc_auc"].rank())
    else:
        pearson_auc = spearman_auc = np.nan
    correlation_rows.append(
        {
            "representation_measure": description,
            "labels": len(values),
            "pearson_with_auc": pearson_auc,
            "spearman_with_auc": spearman_auc,
        }
    )
representation_correlations = pd.DataFrame(correlation_rows).set_index(
    "representation_measure"
)
print(f"Labels with a defined test ROC-AUC: {len(auc_defined)}/{len(label_assessment)}")
print(
    f"Labels retained with at least {MIN_TEST_CLASS_COUNT} positives and negatives: "
    f"{len(auc_labels)}/{len(label_assessment)}"
)
display(
    representation_correlations,
    auc_labels.sort_values("test_roc_auc")[
        [
            "test_roc_auc",
            "test_average_precision",
            "train_positive_count",
            "train_positive_prevalence_pct",
            "positive_feature_coverage_pct",
            "positive_feature_density_pct",
            "mean_feature_prevalence_gap",
        ]
    ].head(20),
)

In [ ]:
plot_metrics = {
    "Positive training samples": "train_positive_count",
    "Unique positive feature profiles": "unique_positive_profiles",
    "Positive feature coverage (%)": "positive_feature_coverage_pct",
    "Positive feature density (%)": "positive_feature_density_pct",
    "Mean positive-negative feature gap": "mean_feature_prevalence_gap",
    "Signal features (gap ≥ 10%)": "signal_features_10pct",
}
fig, axes = plt.subplots(2, 3, figsize=(16, 10), constrained_layout=True)
for axis, (xlabel, column) in zip(axes.flat, plot_metrics.items()):
    values = auc_labels[[column, "test_roc_auc"]].dropna()
    points = axis.scatter(
        values[column],
        values["test_roc_auc"],
        c=auc_labels.loc[values.index, "train_positive_prevalence_pct"],
        cmap="viridis",
        alpha=0.75,
        edgecolor="black",
        linewidth=0.3,
    )
    if values[column].nunique() > 1:
        slope, intercept = np.polyfit(values[column], values["test_roc_auc"], 1)
        x_line = np.linspace(values[column].min(), values[column].max(), 100)
        axis.plot(x_line, slope * x_line + intercept, color="crimson", linestyle="--")
    axis.axhline(0.5, color="grey", linestyle=":")
    axis.set(xlabel=xlabel, ylabel="Test ROC-AUC")
    axis.grid(alpha=0.2)
fig.colorbar(points, ax=axes.ravel().tolist(), label="Positive training prevalence (%)")
fig.suptitle("Positive-label representation versus held-out discrimination", fontsize=14)
plt.show()

Evidence for the hypothesis would be positive correlations between ROC-AUC and positive count, feature coverage, density, profile diversity, or positive-negative feature separation. A weak relationship suggests that label noise, feature relevance, model capacity, or train/test instability may matter more than representation alone. Accuracy should not be used as the main comparison because rare labels can achieve high accuracy by predicting negatives.